In [2]:
import json
import logging
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.signal import hilbert
from tqdm.auto import tqdm

# ============================================================
# CONFIG — modifica qui, poi Kernel → Restart & Run All
# ============================================================

# Metriche per cui costruire i grafi — una directory di output per metrica.
# Ablation: metti tutte e 5 per confronto completo.
METRICS = ["pcc", "abs_pcc", "im_pcc", "wpli", "plv"]

# Metriche usate per il consensus pruning (indipendente da METRICS)
CONSENSUS_METRICS = ["wpli", "plv", "abs_pcc", "im_pcc"]

# Soglia consensus: None = maggioranza (ceil(n/2)); int = fisso
CONSENSUS_K = None

# Top-X% per metric: arco significativo se |valore| >= percentile X
EDGE_THRESHOLD_PCT = 80     # top 20% archi per metrica

# k vicini per iperedge (costruzione matrice di incidenza H)
K_HYPEREDGE = 6

# Salta trial già processati (utile per resume dopo crash)
OVERWRITE = False

# ============================================================
# SETUP — path, logging, mapping parola → label_idx
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("eeg07f")

# Root di progetto
project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve(),
)

CSV_ROOT = project_root / "data" / "raw_csv" / "training_set"
DATA_OUT = project_root / "data"   # graphs_{metric}/ e graphs_pruned_{metric}/ qui sotto

assert CSV_ROOT.exists(), f"CSV_ROOT non trovato: {CSV_ROOT}"
log.info(f"project_root : {project_root}")
log.info(f"CSV_ROOT     : {CSV_ROOT}")

# word → label_idx
_label2idx_path = project_root / "configs" / "label_schemes" / "label2idx.json"
word2label: dict = json.loads(_label2idx_path.read_text())
log.info(f"Vocabolario  : {len(word2label)} parole")

# Verifica metriche configurate
_VALID_METRICS = {"pcc", "abs_pcc", "im_pcc", "wpli", "plv"}
assert all(m in _VALID_METRICS for m in METRICS), f"Metrica non valida in METRICS"
assert all(m in _VALID_METRICS for m in CONSENSUS_METRICS), f"Metrica non valida in CONSENSUS_METRICS"

_eff_k = CONSENSUS_K if CONSENSUS_K is not None else __import__("math").ceil(len(CONSENSUS_METRICS) / 2)
log.info(f"METRICS      : {METRICS}")
log.info(f"Consensus    : {CONSENSUS_METRICS}  k={_eff_k}/{len(CONSENSUS_METRICS)}")
log.info(f"Threshold    : percentile {EDGE_THRESHOLD_PCT} → top {100-EDGE_THRESHOLD_PCT:.0f}%")


12:34:24  INFO      project_root : /home/daniele_u/miralis-hypergraph-imagined-speech
12:34:24  INFO      CSV_ROOT     : /home/daniele_u/miralis-hypergraph-imagined-speech/data/raw_csv/training_set
12:34:24  INFO      Vocabolario  : 110 parole
12:34:24  INFO      METRICS      : ['pcc', 'abs_pcc', 'im_pcc', 'wpli', 'plv']
12:34:24  INFO      Consensus    : ['wpli', 'plv', 'abs_pcc', 'im_pcc']  k=2/4
12:34:24  INFO      Threshold    : percentile 80 → top 20%


In [3]:
import torch
import numpy as np

# Scegliamo un trial a caso
trial_path = "P000_S001/trial_000.pt"
base = project_root / "data"

# Carichiamo due metriche diverse
g_pcc = torch.load(base / "graphs_pruned_abs_pcc" / trial_path, weights_only=False)
g_wpli = torch.load(base / "graphs_pruned_wpli" / trial_path, weights_only=False)
h_pcc = torch.load(base / "hypergraphs_pruned_abs_pcc" / trial_path, weights_only=False)
h_wpli = torch.load(base / "hypergraphs_pruned_wpli" / trial_path, weights_only=False)

print(f"--- ANALISI TRIAL: {trial_path} ---")
# Verifica topologia
same_topo = torch.equal(g_pcc['edge_index'], g_wpli['edge_index'])
print(f"1. Stessi archi (edge_index)? {same_topo} (N={g_pcc['edge_index'].shape[1]})")

# Verifica pesi
same_weights = torch.allclose(g_pcc['edge_attr'], g_wpli['edge_attr'])
print(f"2. Stessi pesi (edge_attr)?   {same_weights}")
if not same_weights:
    print(f"   Esempio peso PCC:  {g_pcc['edge_attr'][0].item():.4f}")
    print(f"   Esempio peso wPLI: {g_wpli['edge_attr'][0].item():.4f}")

# Verifica ipergrafi
same_h = torch.equal(h_pcc['H'], h_wpli['H'])
print(f"3. Ipergrafo (H) identico?  {same_h}")
if not same_h:
    diff = (h_pcc['H'] != h_wpli['H']).sum().item()
    print(f"   Differenze in H: {diff} elementi su {h_pcc['H'].numel()}")


--- ANALISI TRIAL: P000_S001/trial_000.pt ---
1. Stessi archi (edge_index)? True (N=1162)
2. Stessi pesi (edge_attr)?   False
   Esempio peso PCC:  0.0617
   Esempio peso wPLI: 0.4638
3. Ipergrafo (H) identico?  False
   Differenze in H: 546 elementi su 3721
